# 05 – Feature Engineering: Variables de Empleo

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Construir nuevas variables que capturen características del empleo (tipo, intensidad, sector) relevantes para predecir la condición de desocupación.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

FE_DIR = os.path.join('..', 'data', 'feature_engineering')
try:
    df = pd.read_csv(os.path.join(FE_DIR, 'epen_fe_income.csv'))
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'edad': np.random.randint(14, 70, n),
        'horas_trabajadas': np.random.randint(0, 60, n),
        'tipo_empleo_cod': np.random.choice([-1, 0, 1], n),
        'ingreso_mensual': np.random.exponential(8000, n).round(2),
        'target_desocupado': np.random.choice([0, 1], n, p=[0.50, 0.50]),
    })

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

Dataset cargado: 1,000 filas × 5 columnas


## 1. Indicador de jornada completa (≥40 horas/semana)

In [2]:
if 'horas_trabajadas' in df.columns:
    df['jornada_completa'] = (df['horas_trabajadas'] >= 40).astype(int)
    print('Variable creada: jornada_completa')
    print(df['jornada_completa'].value_counts())

Variable creada: jornada_completa
jornada_completa
0    666
1    334
Name: count, dtype: int64


## 2. Indicador de subempleo (< 15 horas/semana y con empleo)

In [3]:
if 'horas_trabajadas' in df.columns and 'tipo_empleo_cod' in df.columns:
    # tipo_empleo_cod: 1=Formal, 0=Informal, -1=Sin empleo
    tiene_empleo = df['tipo_empleo_cod'].isin([0, 1])
    df['subempleado'] = ((df['horas_trabajadas'] < 15) & tiene_empleo).astype(int)
    print('Variable creada: subempleado')
    print(df['subempleado'].value_counts())

Variable creada: subempleado
subempleado
0    831
1    169
Name: count, dtype: int64


## 3. Indicador de empleo formal

In [4]:
if 'tipo_empleo_cod' in df.columns:
    df['es_formal'] = (df['tipo_empleo_cod'] == 1).astype(int)
    print('Variable creada: es_formal')
    print(df['es_formal'].value_counts())

Variable creada: es_formal
es_formal
0    694
1    306
Name: count, dtype: int64


## 4. Categoría de intensidad laboral

In [5]:
if 'horas_trabajadas' in df.columns:
    bins = [-1, 0, 14, 39, 168]
    labels_int = [0, 1, 2, 3]  # Sin trabajo, Parcial, Completa, Extra
    df['intensidad_laboral'] = pd.cut(df['horas_trabajadas'], bins=bins, labels=labels_int).astype(int)
    print('Variable creada: intensidad_laboral')
    print(df['intensidad_laboral'].value_counts().sort_index())

Variable creada: intensidad_laboral
intensidad_laboral
0     16
1    227
2    423
3    334
Name: count, dtype: int64


In [6]:
os.makedirs(FE_DIR, exist_ok=True)
df.to_csv(os.path.join(FE_DIR, 'epen_fe_employment.csv'), index=False)
nuevas = ['jornada_completa', 'subempleado', 'es_formal', 'intensidad_laboral']
print('Nuevas variables de empleo:', [c for c in nuevas if c in df.columns])
print('Dataset guardado: epen_fe_employment.csv')

Nuevas variables de empleo: ['jornada_completa', 'subempleado', 'es_formal', 'intensidad_laboral']
Dataset guardado: epen_fe_employment.csv
